# Example 1 — Beta Sensitivity Sweep

Reproduces **Figure 1** of Alberts & Bilionis (2023): PIFT posterior variance for the 1D steady-state heat equation collapses as β → ∞. Sweep β ∈ {1, 10, 100, 1000} with non-zero Dirichlet BCs and source q(x)=exp(-x), mixed Fourier basis with BC embedding (Eqs. 35–36).

In [ ]:
# Colab bootstrap — installs CUDA-enabled JAX. No-op locally.
import os, sys
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    %cd /content
    !nvidia-smi -L || echo 'No GPU detected — Runtime > Change runtime type > GPU'
    !rm -rf /content/pift-od-il-inverse-problems
    !git clone https://github.com/cmhobbs96/pift-od-il-inverse-problems.git /content/pift-od-il-inverse-problems
    assert Path('/content/pift-od-il-inverse-problems/pyproject.toml').exists()
    %cd /content/pift-od-il-inverse-problems
    !pip install -q --upgrade pip
    !pip install -q -e .
    !pip install -q --upgrade "jax[cuda12]"
    os.environ['PIFT_FORCE_BACKEND'] = 'local'
    os.environ['PIFT_FORCE_DEVICE'] = 'gpu'
    os.environ['JAX_PLATFORMS'] = 'cuda'
    os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
    print('Colab bootstrap complete.')
else:
    print('Not on Colab — bootstrap skipped.')

In [ ]:
# Verify JAX sees CUDA when on Colab
import jax
print('JAX version :', jax.__version__)
print('JAX backend :', jax.default_backend())
print('JAX devices :', jax.devices())
if ON_COLAB:
    assert any(d.platform == 'gpu' for d in jax.devices()), 'CUDA GPU not visible to JAX'
    print('CUDA enabled ✓')

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
DEVICE = 'gpu' if ON_COLAB else 'cpu'
print('Repo root:', ROOT, '| device_preference:', DEVICE)

In [ ]:
from src.pipelines.phase_b_beta_sweep import run_phase_b_beta_sweep
result = run_phase_b_beta_sweep(device_preference=DEVICE)
print('status:', result['status'], '| runtime (s):', round(result['runtime_sec'], 1))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
for path in result['artifacts']:
    if str(path).endswith('.png'):
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(mpimg.imread(path)); ax.set_title(Path(path).name); ax.axis('off')
        plt.show()

In [ ]:
print('Posterior variance scaling vs beta:')
for beta, var in zip([r['beta'] for r in result['beta_results']], result['variance_scaling']):
    print(f'  beta = {beta:8.1f}    mean posterior var = {var:.3e}')

**Expected:** posterior variance scales like 1/β — strong-physics limit recovers the deterministic Klein–Gordon solution.